# 05 — ToN-IoT Model Evaluation

**Goal:** Compare the five trained models and investigate their errors.

**Input:** Saved Core10 models, training reports, and the frozen validation split.

**Output:** Validation tables, two graphs and one JSON report.

**Rule:** No training and no test data access.

In [ ]:
from pathlib import Path
import hashlib, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
VALIDATION = ROOT / 'data/splits/ton_iot/ton_iot_core10_validation_v3.csv.gz'
SPLIT_REPORT = ROOT / 'reports/audits/ton_iot_v3_split_report.json'
MANIFEST = ROOT / 'data/splits/ton_iot/ton_iot_core10_signature_split_manifest_v3.csv'
MODELS_DIR = ROOT / 'models/ton_iot_v3'
AUDITS = ROOT / 'reports/audits'
METRICS = ROOT / 'reports/metrics'
FIGURES = ROOT / 'reports/figures'
MODEL_NAMES = ['decision_tree', 'random_forest', 'extra_trees', 'lightgbm', 'adaboost']
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown', '<missing>'}

for file in [VALIDATION, SPLIT_REPORT, MANIFEST]:
    if not file.is_file():
        raise FileNotFoundError(f'Missing Notebook 03 result: {file}')
report = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
if report.get('dataset') != 'ToN-IoT' or report.get('test_scored', False):
    raise ValueError('Invalid ToN-IoT split report or test split already scored.')
if report.get('normalized_fingerprint_overlap_groups') != 0:
    raise ValueError('Feature fingerprints cross partitions.')
if report.get('total_rows_preserved') != 211043:
    raise ValueError('Dataset total changed: do not silently resplit.')
counts = report['counts']
if {p: int(counts[p]['rows']) for p in ('train','validation','test')} != {
        'train':147730,'validation':31657,'test':31656}:
    raise ValueError('Unexpected train/validation/test row counts.')
sha = hashlib.sha256()
with MANIFEST.open('rb') as fh:
    for block in iter(lambda: fh.read(8*1024*1024), b''):
        sha.update(block)
if sha.hexdigest() != report['manifest_sha256']:
    raise ValueError('Frozen split manifest has changed.')
FEATURES = list(report['model_features'])
CAT = list(report['categorical_features'])
NUM = [c for c in FEATURES if c not in CAT]
if len(FEATURES) != 10 or len(CAT) != 3 or len(NUM) != 7:
    raise ValueError('Unexpected Core10 feature schema.')
for folder in [METRICS, FIGURES, AUDITS]:
    folder.mkdir(parents=True, exist_ok=True)
print('ToN-IoT | Validation:', counts['validation']['rows'], '| Core10 features:', len(FEATURES))
print('Saved models only; NO retraining and NO test-set access.')

## 1. Compare the saved results

Check each model's training report and summarize detection and laptop resources.

In [ ]:
records = []
for name in MODEL_NAMES:
    path = AUDITS / f'ton_iot_v3_{name}_training.json'
    model_path = MODELS_DIR / f'{name}_core10_v3.joblib'
    if not path.is_file() or not model_path.is_file():
        raise FileNotFoundError(f'Missing saved result for {name}. Complete Notebook 04 first.')
    entry = json.loads(path.read_text(encoding='utf-8'))
    if (entry.get('model') != name or
        entry.get('frozen_manifest_sha256') != report['manifest_sha256'] or
        list(entry.get('features', [])) != FEATURES or
        int(entry['metrics']['training_rows']) != 147730 or
        int(entry['metrics']['validation_rows']) != 31657 or
        entry.get('test_scored', False)):
        raise ValueError(f'{name}: model report does not match current training experiment.')
    records.append(entry)

summary = pd.DataFrame([{**x['metrics'], **x['resources']} for x in records])
summary['f1_pct'] = summary['f1_attack'] * 100
summary['fpr_pct'] = summary['false_positive_rate'] * 100
summary['missed_attacks'] = summary['fn'].astype(int)
summary['false_alarms'] = summary['fp'].astype(int)
columns = ['model','f1_pct','fpr_pct','false_alarms','missed_attacks',
           'training_seconds','single_flow_p95_ms','model_size_mib',
           'train_process_cpu_percent','train_peak_process_ram_mib']
display(summary[columns].round(3))
print('Resource numbers are from the Notebook 04 laptop runs, not Raspberry Pi.')

## 2. Evaluate attacks by type

Use only the validation data and saved models. Check the attack categories that each model misses.

In [ ]:
frame = pd.read_csv(VALIDATION, dtype=str, keep_default_na=False, low_memory=False)
expected_cols = ['csv_row_index_0based'] + FEATURES + ['label', 'type']
if frame.columns.tolist() != expected_cols or len(frame) != 31657:
    raise ValueError('Validation file schema or size changed.')
if frame['csv_row_index_0based'].duplicated().any():
    raise ValueError('Duplicate original row identifiers.')
if not frame['label'].isin(['0','1']).all():
    raise ValueError('Unknown binary target labels.')
y = frame['label'].astype(np.uint8).to_numpy()
if int((y == 0).sum()) != 7500 or int((y == 1).sum()) != 24157:
    raise ValueError('Validation label counts changed.')
attack_type = frame['type'].astype(str).str.strip()
if ((y == 0) & ~attack_type.str.casefold().eq('normal')).any():
    raise ValueError('Normal records have unexpected attack type.')
if ((y == 1) & attack_type.str.casefold().eq('normal')).any():
    raise ValueError('Attack records marked as normal type.')

service_missing = frame['service'].astype(str).str.strip().str.casefold().isin(MISSING).to_numpy()
protocol = frame['proto'].astype(str).str.strip().str.casefold().to_numpy()

def clean_features(raw):
    data = raw[FEATURES].copy()
    for col in NUM:
        s = data[col].astype(str).str.strip()
        s = s.mask(s.str.casefold().isin(MISSING))
        data[col] = pd.to_numeric(s, errors='coerce').replace([np.inf,-np.inf], np.nan).astype(np.float32)
    for col in CAT:
        s = data[col].astype(str).str.strip()
        data[col] = s.mask(s.str.casefold().isin(MISSING), '__MISSING__')
    return data
X = clean_features(frame)
category_rows, service_rows, protocol_rows = [], [], []
predictions = {}
for entry in records:
    name = entry['model']
    pipeline = joblib.load(MODELS_DIR / f'{name}_core10_v3.joblib')
    if (not hasattr(pipeline, 'named_steps') or
        not {'preprocessing','classifier'}.issubset(pipeline.named_steps)):
        raise ValueError(f'{name}: saved artifact is not the expected pipeline.')
    pred = np.asarray(pipeline.predict(X), dtype=np.uint8)
    if len(pred) != len(y) or not np.isin(pred, [0,1]).all():
        raise ValueError(f'{name}: invalid validation predictions.')
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(y, pred, labels=[0,1]).ravel())
    old = entry['metrics']
    if (tn,fp,fn,tp) != tuple(int(old[k]) for k in ['tn','fp','fn','tp']):
        raise ValueError(f'{name}: predictions differ from Notebook 04. Check model/data versions.')
    predictions[name] = pred
    attack_mask = y == 1
    for kind in sorted(attack_type[attack_mask].unique()):
        mask = attack_mask & attack_type.eq(kind).to_numpy()
        total = int(mask.sum()); detected = int((pred[mask] == 1).sum())
        category_rows.append({'model':name,'attack_type':kind,'attack_count':total,
            'detected':detected,'missed':total-detected,
            'recall_pct':round(100*detected/total,2)})
    for status, missing_bool in [('missing',True), ('present',False)]:
        for label, label_value in [('Normal',0),('Attack',1)]:
            mask = (service_missing == missing_bool) & (y == label_value)
            total = int(mask.sum()); errors = int((pred[mask] != label_value).sum())
            service_rows.append({'model':name,'service':status,'actual_class':label,
                'records':total,'errors':errors,'error_pct':round(100*errors/total,2) if total else np.nan,
                'metric':'FPR' if label_value == 0 else 'FNR'})
    for p in sorted(set(protocol)):
        mask = (protocol == p)
        n = int(mask.sum())
        protocol_rows.append({'model':name,'protocol':p,'records':n,
            'errors':int((pred[mask] != y[mask]).sum()),
            'error_pct':round(100*(pred[mask] != y[mask]).mean(),2) if n else np.nan})
    del pipeline, pred
    gc.collect()

category_result = pd.DataFrame(category_rows)
service_result = pd.DataFrame(service_rows)
protocol_result = pd.DataFrame(protocol_rows)
print('Attack recall by type (top three models):')
display(category_result[category_result['model'].isin(['lightgbm','random_forest','decision_tree'])]
        .sort_values(['attack_type','model']).reset_index(drop=True))

## 3. Check the missing `service` field

Compare prediction errors when `service` is missing versus present. The difference may reveal a dataset-specific shortcut.

In [ ]:
print('Service availability by actual class:')
display(pd.crosstab(
    pd.Series(np.where(y == 1,'Attack','Normal'), name='Actual'),
    pd.Series(np.where(service_missing,'Missing','Present'), name='Service'),
    margins=True
))
print('Error rates by service availability (top three models):')
display(service_result[service_result['model'].isin(['lightgbm','random_forest','decision_tree'])]
        .sort_values(['model','actual_class','service']).reset_index(drop=True))
print('Protocol groups (diagnostic only):')
display(protocol_result[protocol_result['model'] == 'lightgbm'].sort_values('records', ascending=False))

## 4. Graphs and saved results

Show detection trade-offs and LightGBM's recall by attack type. Save compact CSV and JSON results.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for _, row in summary.iterrows():
    ax.scatter(row['fpr_pct'], row['f1_pct'], s=75)
    ax.annotate(row['model'].replace('_',' ').title(),
                (row['fpr_pct'],row['f1_pct']),xytext=(5,6),textcoords='offset points',fontsize=9)
ax.set_xlabel('False-positive rate (%)')
ax.set_ylabel('Attack F1-score (%)')
ax.set_title('ToN-IoT — Detection versus false alarms (validation)')
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES / 'ton_iot_v3_evaluation_tradeoff.png', dpi=180, bbox_inches='tight')
plt.show();plt.close(fig)

best = category_result[category_result['model'] == 'lightgbm'].sort_values('recall_pct')
fig, ax = plt.subplots(figsize=(9,5))
ax.barh(best['attack_type'],best['recall_pct'])
ax.set_xlim(0, 105)
ax.set_xlabel('Attack recall (%)')
ax.set_title('ToN-IoT — LightGBM recall by attack type (validation)')
ax.grid(axis='x',alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES / 'ton_iot_v3_attack_type_recall.png',dpi=180,bbox_inches='tight')
plt.show();plt.close(fig)

paths = {
    'model_evaluation':'ton_iot_v3_model_evaluation.csv',
    'attack_type_recall':'ton_iot_v3_attack_type_recall.csv',
    'service_missingness':'ton_iot_v3_service_missingness_evaluation.csv',
    'protocol_groups':'ton_iot_v3_protocol_evaluation.csv',
}
summary.to_csv(METRICS / paths['model_evaluation'],index=False)
category_result.to_csv(METRICS / paths['attack_type_recall'],index=False)
service_result.to_csv(METRICS / paths['service_missingness'],index=False)
protocol_result.to_csv(METRICS / paths['protocol_groups'],index=False)

report_out = {
    'dataset':'ToN-IoT', 'experiment':'Core10 V3 validation error analysis',
    'source_validation_file':VALIDATION.name,
    'frozen_manifest_sha256':report['manifest_sha256'],
    'features':FEATURES, 'validation_rows':int(len(y)),
    'normal_rows':int((y==0).sum()), 'attack_rows':int((y==1).sum()),
    'models':MODEL_NAMES, 'test_scored':False,
    'summary_csv_files':paths,
    'figure_files':['ton_iot_v3_evaluation_tradeoff.png','ton_iot_v3_attack_type_recall.png'],
    'limitations':[
        'Validation-only results; held-out test data was not opened.',
        'Group-disjoint feature fingerprints do not establish session or capture independence.',
        'A high service placeholder rate can create dataset-specific shortcuts.',
        'Laptop resources were measured in Notebook 04, not on a Raspberry Pi.',
        'Attack types are dataset labels, not proven APT campaign-level detection.'
    ]
}
report_file = AUDITS / 'ton_iot_v3_model_evaluation.json'
report_file.write_text(json.dumps(report_out,indent=2),encoding='utf-8')
print('Saved:', report_file)
print('Saved: 4 CSV metrics and 2 PNG graphs. No training and no test evaluation.')